# Smart Rental Price Prediction
**A complete beginner-friendly ML project**

Steps covered:
1. Load & clean data
2. EDA with visualizations
3. Feature engineering & encoding
4. Train Linear Regression
5. Evaluate (R², MAE)
6. Compare with Decision Tree
7. Overpriced / Fairly Priced checker

In [ ]:
# Install required libraries if needed:
# !pip install pandas numpy matplotlib scikit-learn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import os
os.makedirs('../plots', exist_ok=True)
print('✅ All libraries loaded!')

## Step 1 — Load & Clean Data
Load the CSV and handle missing values, impossible numbers, etc.

In [ ]:
df = pd.read_csv('../data/rental_data.csv')
print('Raw shape:', df.shape)
print('\nMissing values:\n', df.isnull().sum())
df.head()

In [ ]:
# Clean the data
df.dropna(subset=['Rent', 'Size'], inplace=True)
df['BHK'].fillna(df['BHK'].mode()[0], inplace=True)
df['Furnishing Status'].fillna('Unfurnished', inplace=True)
df = df[(df['Rent'] > 0) & (df['Size'] > 0) & (df['BHK'] > 0)]
df.reset_index(drop=True, inplace=True)
print('Clean shape:', df.shape)
df.describe()

## Step 2 — EDA (Exploratory Data Analysis)
Visualize the data to understand patterns before modeling.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# Rent distribution
axes[0,0].hist(df['Rent'], bins=20, color='steelblue', edgecolor='white')
axes[0,0].set_title('Rent Distribution')
axes[0,0].set_xlabel('Rent (₹)')
axes[0,0].set_ylabel('Count')

# Average rent by city
avg_city = df.groupby('City')['Rent'].mean().sort_values(ascending=False)
avg_city.plot(kind='bar', ax=axes[0,1], color='coral', edgecolor='white')
axes[0,1].set_title('Avg Rent by City')
axes[0,1].tick_params(axis='x', rotation=30)

# Rent vs Size
axes[1,0].scatter(df['Size'], df['Rent'], alpha=0.5, color='teal')
axes[1,0].set_title('Rent vs Size (sq ft)')
axes[1,0].set_xlabel('Size (sq ft)')
axes[1,0].set_ylabel('Rent (₹)')

# Rent by furnishing
avg_furnish = df.groupby('Furnishing Status')['Rent'].mean().sort_values(ascending=False)
avg_furnish.plot(kind='bar', ax=axes[1,1], color='mediumpurple', edgecolor='white')
axes[1,1].set_title('Avg Rent by Furnishing')
axes[1,1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('../plots/eda_overview.png', dpi=100)
plt.show()
print('Saved: plots/eda_overview.png')

## Step 3 — Feature Engineering & Encoding
Create new features and convert text to numbers for ML.

In [ ]:
# New feature: price per square foot
df['Price_per_sqft'] = df['Rent'] / df['Size']

# Label encode city names
le_city = LabelEncoder()
df['City_encoded'] = le_city.fit_transform(df['City'])

# Label encode furnishing status
le_furnish = LabelEncoder()
df['Furnishing_encoded'] = le_furnish.fit_transform(df['Furnishing Status'])

print('City mapping   :', dict(zip(le_city.classes_, le_city.transform(le_city.classes_).tolist())))
print('Furnish mapping:', dict(zip(le_furnish.classes_, le_furnish.transform(le_furnish.classes_).tolist())))

# Define features and target
features = ['BHK', 'Size', 'City_encoded', 'Furnishing_encoded', 'Price_per_sqft']
X = df[features]
y = df['Rent']

print(f'\nX shape: {X.shape} | y shape: {y.shape}')
X.head()

## Steps 4 & 5 — Train & Evaluate Linear Regression

In [ ]:
# Split: 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {len(X_train)} samples | Test: {len(X_test)} samples')

# Train
lr = LinearRegression()
lr.fit(X_train, y_train)

# Predict and evaluate
lr_pred = lr.predict(X_test)
lr_r2   = r2_score(y_test, lr_pred)
lr_mae  = mean_absolute_error(y_test, lr_pred)

print(f'\nLinear Regression:')
print(f'  R² Score : {lr_r2:.4f}  (1.0 = perfect)')
print(f'  MAE      : ₹{lr_mae:,.0f}  (average error)')

In [ ]:
# Actual vs Predicted plot
plt.figure(figsize=(6, 5))
plt.scatter(y_test, lr_pred, alpha=0.6, color='steelblue')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', label='Perfect')
plt.xlabel('Actual Rent (₹)'); plt.ylabel('Predicted Rent (₹)')
plt.title('Actual vs Predicted — Linear Regression')
plt.legend(); plt.tight_layout()
plt.savefig('../plots/actual_vs_predicted_lr.png', dpi=100)
plt.show()

## Step 6 — Compare with Decision Tree

In [ ]:
dt = DecisionTreeRegressor(max_depth=6, random_state=42)
dt.fit(X_train, y_train)
dt_pred = dt.predict(X_test)
dt_r2   = r2_score(y_test, dt_pred)
dt_mae  = mean_absolute_error(y_test, dt_pred)

print(f'Decision Tree:')
print(f'  R² Score : {dt_r2:.4f}')
print(f'  MAE      : ₹{dt_mae:,.0f}')

# Comparison chart
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
names = ['Linear\nRegression', 'Decision\nTree']

bars = axes[0].bar(names, [lr_r2, dt_r2], color=['steelblue','coral'], edgecolor='white', width=0.5)
axes[0].set_title('R² Score (higher = better)'); axes[0].set_ylim(0, 1.1)
for b, v in zip(bars, [lr_r2, dt_r2]):
    axes[0].text(b.get_x()+b.get_width()/2, v+0.02, f'{v:.3f}', ha='center', fontsize=11)

bars2 = axes[1].bar(names, [lr_mae, dt_mae], color=['steelblue','coral'], edgecolor='white', width=0.5)
axes[1].set_title('MAE (lower = better)')
for b, v in zip(bars2, [lr_mae, dt_mae]):
    axes[1].text(b.get_x()+b.get_width()/2, v+200, f'₹{v:,.0f}', ha='center', fontsize=10)

plt.suptitle('Model Comparison', fontsize=14)
plt.tight_layout()
plt.savefig('../plots/model_comparison.png', dpi=100)
plt.show()

## Step 7 — Overpriced / Fairly Priced Checker
Enter any property details to check if it's a good deal!

In [ ]:
def check_pricing(size, bhk, city_name, furnishing_status, actual_rent):
    city_enc    = le_city.transform([city_name])[0]
    furnish_enc = le_furnish.transform([furnishing_status])[0]
    pps = actual_rent / size
    inp = pd.DataFrame([[bhk, size, city_enc, furnish_enc, pps]], columns=features)
    predicted = lr.predict(inp)[0]
    diff = actual_rent - predicted
    pct  = (diff / predicted) * 100
    
    print(f'\nActual: ₹{actual_rent:,} | Predicted: ₹{predicted:,.0f} | Diff: {pct:+.1f}%')
    if pct > 15:    print('Verdict: ⚠️  OVERPRICED')
    elif pct < -15: print('Verdict: ✅  UNDERPRICED — Great deal!')
    else:           print('Verdict: ✅  FAIRLY PRICED')

print('Valid cities   :', list(le_city.classes_))
print('Valid furnishing:', list(le_furnish.classes_))

In [ ]:
# Try different properties!
check_pricing(size=900,  bhk=2, city_name='Delhi',     furnishing_status='Furnished',      actual_rent=45000)
check_pricing(size=1200, bhk=2, city_name='Mumbai',    furnishing_status='Semi-Furnished', actual_rent=27000)
check_pricing(size=1500, bhk=3, city_name='Bangalore', furnishing_status='Furnished',      actual_rent=40000)

In [ ]:
# ✏️ Try your own property here:
check_pricing(
    size=1000,
    bhk=2,
    city_name='Chennai',        # Change this
    furnishing_status='Furnished',  # Change this
    actual_rent=25000           # Change this
)